# 01 — Preprocessing

**SolarVision — Feature 2: Solar Power Forecasting**

This notebook prepares the Laghouat 2022 PV + dust dataset for model training.

### Principles
- Use **only** columns present in `Laghouat_2022_final.csv`
- No external weather APIs, no invented sensors
- Strict **temporal** logic: lags and rolling stats use past values only
- Two modelling targets are created:
  1. **Nowcast** — same-hour `P_dusty_W` (comparable to Seghir et al. MATLAB pipeline)
  2. **24h-ahead** — `P_dusty_W` shifted −24 h (operational forecast for SolarVision)

### Known data limits (documented, not hidden)
- Single site (Laghouat, Algeria), Feb–Aug 2022 only
- Dust density is **reconstructed** (PCHIP) between sparse gravimetric samples
- `P_dusty` and `P_clean` come from two side-by-side modules; not identical optics
- Morning building-shadow effects exist in the raw series
- No plant-scale multi-inverter data → cleaning economics stay scenario-based

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- paths (edit if needed) ---
RAW_PATH = Path("Laghouat_2022_final.csv")          # put CSV next to this notebook
OUT_DIR  = Path("processed_data")
OUT_DIR.mkdir(exist_ok=True)

print("Raw path exists:", RAW_PATH.exists())

Raw path exists: True


## 1. Load & basic checks

In [2]:
df = pd.read_csv(RAW_PATH)
df["datetime"] = pd.to_datetime(df["datetime"], dayfirst=True, format="%d/%m/%Y %H:%M")
df = df.sort_values("datetime").reset_index(drop=True)
df = df.set_index("datetime")

print("Shape:", df.shape)
print("Range:", df.index.min(), "→", df.index.max())
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing %:")
print((df.isna().mean() * 100).round(2).sort_values(ascending=False).head(15))

Shape: (4368, 13)
Range: 2022-02-17 00:00:00 → 2022-08-17 23:00:00

Columns:
['dust_density_measured_gm2', 'dust_density_reconstructed_PCHIP_gm2', 'temp_amb_C', 'T_cell_C', 'G_POA_Wm2', 'RH_pct', 'wind_speed_ms', 'cloud_cover_pct', 'precipitation_mm', 'P_clean_W', 'P_dusty_W', 'P_clean_norm', 'P_dusty_norm']

Missing %:
dust_density_measured_gm2               98.9
dust_density_reconstructed_PCHIP_gm2     0.0
temp_amb_C                               0.0
T_cell_C                                 0.0
G_POA_Wm2                                0.0
RH_pct                                   0.0
wind_speed_ms                            0.0
cloud_cover_pct                          0.0
precipitation_mm                         0.0
P_clean_W                                0.0
P_dusty_W                                0.0
P_clean_norm                             0.0
P_dusty_norm                             0.0
dtype: float64


## 2. Column selection (dataset only)

We keep physical and reconstructed fields that exist in the official release.

In [3]:
# Expected columns from Laghouat technical note (subset may vary slightly by version)
CANDIDATES = [
    "P_dusty_W", "P_clean_W", "P_clean_norm",
    "G_POA_Wm2", "temp_amb_C", "T_cell_C", "RH_pct",
    "wind_speed_ms", "cloud_cover_pct", "precipitation_mm",
    "dust_density_reconstructed_PCHIP_gm2",
    "dust_density_measured_gm2",
]

available = [c for c in CANDIDATES if c in df.columns]
missing_cols = [c for c in CANDIDATES if c not in df.columns]
print("Available:", available)
print("Missing from this file:", missing_cols)

work = df[available].copy()

# Numeric coercion
for c in work.columns:
    work[c] = pd.to_numeric(work[c], errors="coerce")

# Target must exist
assert "P_dusty_W" in work.columns, "P_dusty_W is required"

Available: ['P_dusty_W', 'P_clean_W', 'P_clean_norm', 'G_POA_Wm2', 'temp_amb_C', 'T_cell_C', 'RH_pct', 'wind_speed_ms', 'cloud_cover_pct', 'precipitation_mm', 'dust_density_reconstructed_PCHIP_gm2', 'dust_density_measured_gm2']
Missing from this file: []


## 3. Minimal cleaning

- Clip impossible negatives on power / irradiance to 0
- Keep NaNs in rarely measured gravimetric dust; models will not require that column
- No row deletion beyond what lag creation naturally drops

In [4]:
for c in ["P_dusty_W", "P_clean_W", "G_POA_Wm2"]:
    if c in work.columns:
        work[c] = work[c].clip(lower=0)

# Daytime flag from irradiance when available, else from power
if "G_POA_Wm2" in work.columns:
    work["is_daytime"] = (work["G_POA_Wm2"] > 5).astype(int)
else:
    work["is_daytime"] = (work["P_dusty_W"] > 0.5).astype(int)

print(work.describe().T[["count", "mean", "min", "max"]].round(2))

                                       count    mean  min      max
P_dusty_W                             4368.0   13.79  0.0    77.03
P_clean_W                             4368.0   16.22  0.0    80.29
P_clean_norm                          4368.0    0.20  0.0     1.00
G_POA_Wm2                             4368.0  216.56  0.0  1052.64
temp_amb_C                            4368.0   23.15 -0.6    42.30
T_cell_C                              4368.0   29.91 -0.6    72.00
RH_pct                                4368.0   33.27  5.0   100.00
wind_speed_ms                         4368.0    4.27  0.0    14.20
cloud_cover_pct                       4368.0   34.00  0.0   100.00
precipitation_mm                      4368.0    0.02  0.0     6.00
dust_density_reconstructed_PCHIP_gm2  4368.0   14.60  0.0    22.54
dust_density_measured_gm2               48.0   14.42  0.0    22.54
is_daytime                            4368.0    0.53  0.0     1.00


## 4. Time features (no leakage)

In [5]:
idx = work.index
work["hour"] = idx.hour
work["dayofweek"] = idx.dayofweek
work["month"] = idx.month
work["dayofyear"] = idx.dayofyear

work["hour_sin"] = np.sin(2 * np.pi * work["hour"] / 24)
work["hour_cos"] = np.cos(2 * np.pi * work["hour"] / 24)
work["doy_sin"]  = np.sin(2 * np.pi * work["dayofyear"] / 365)
work["doy_cos"]  = np.cos(2 * np.pi * work["dayofyear"] / 365)

## 5. Lag & rolling features (past only)

Aligned with lessons from the MATLAB pipeline and operational forecasting needs.

In [6]:
def add_lags(frame, col, lags):
    for L in lags:
        frame[f"{col}_lag_{L}h"] = frame[col].shift(L)
    return frame

def add_rolling(frame, col, windows):
    # trailing windows → no future leak
    for w in windows:
        frame[f"{col}_roll_mean_{w}h"] = frame[col].rolling(w, min_periods=1).mean()
        frame[f"{col}_roll_max_{w}h"]  = frame[col].rolling(w, min_periods=1).max()
    return frame

# Power lags (critical for 24h-ahead and strong persistence structure)
work = add_lags(work, "P_dusty_W", [1, 3, 6, 12, 24, 48, 72])
work = add_rolling(work, "P_dusty_W", [3, 6, 12, 24])

# Dust (reconstructed series is dense; still lag it for dynamics)
dust_col = "dust_density_reconstructed_PCHIP_gm2"
if dust_col in work.columns:
    work = add_lags(work, dust_col, [24])
    work = add_rolling(work, dust_col, [24])
    work["dust_sq"] = work[dust_col] ** 2   # nonlinear term used in Seghir pipeline

# Irradiance / temperature short history
if "G_POA_Wm2" in work.columns:
    work = add_lags(work, "G_POA_Wm2", [1, 3, 6, 24])
    work = add_rolling(work, "G_POA_Wm2", [6, 24])
if "temp_amb_C" in work.columns:
    work = add_lags(work, "temp_amb_C", [1, 24])

# Same-hour power yesterday (explicit persistence feature)
work["P_dusty_same_hour_yesterday"] = work["P_dusty_W"].shift(24)

## 6. Targets

- `target_nowcast` = current `P_dusty_W`
- `target_24h` = power 24 hours **ahead** (shift −24 on the series when building supervised rows)

In [7]:
work["target_nowcast"] = work["P_dusty_W"]
work["target_24h"] = work["P_dusty_W"].shift(-24)

# Optional clean-power gap (soiling proxy) — descriptive only, not used as a forced label
if "P_clean_W" in work.columns:
    work["soiling_gap_W"] = (work["P_clean_W"] - work["P_dusty_W"]).clip(lower=0)
    work["soiling_gap_pct"] = np.where(
        work["P_clean_W"] > 1,
        work["soiling_gap_W"] / work["P_clean_W"] * 100,
        np.nan,
    )

## 7. Feature lists & chronological split markers

70% train / 15% val / 15% test — **time-ordered**, never shuffled.

In [8]:
# Drop rows that cannot form 24h target or long lags
model_df = work.dropna(subset=["target_24h", "P_dusty_W_lag_72h"]).copy()

n = len(model_df)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

model_df["split"] = "test"
model_df.iloc[:n_train, model_df.columns.get_loc("split")] = "train"
model_df.iloc[n_train:n_train + n_val, model_df.columns.get_loc("split")] = "val"

print(model_df["split"].value_counts())
print("Train:", model_df.index.min(), "→", model_df.loc[model_df.split=="train"].index.max())
print("Val:  ", model_df.loc[model_df.split=="val"].index.min(), "→", model_df.loc[model_df.split=="val"].index.max())
print("Test: ", model_df.loc[model_df.split=="test"].index.min(), "→", model_df.index.max())

split
train    2990
test      642
val       640
Name: count, dtype: int64
Train: 2022-02-20 00:00:00 → 2022-06-24 13:00:00
Val:   2022-06-24 14:00:00 → 2022-07-21 05:00:00
Test:  2022-07-21 06:00:00 → 2022-08-16 23:00:00


In [9]:
EXCLUDE = {
    "target_nowcast", "target_24h", "split",
    "P_dusty_W",  # raw target leakage if left in features for nowcast carelessly
    "soiling_gap_W", "soiling_gap_pct",
    "dust_density_measured_gm2",  # sparse; reconstructed used instead
}

# For NOWCAST: may use concurrent weather/dust, but NOT concurrent P_dusty_W
features_nowcast = [
    c for c in model_df.columns
    if c not in EXCLUDE
    and not c.startswith("target_")
    and model_df[c].dtype != "O"
]

# For 24H-AHEAD: drop any feature that could encode "current power" too directly
# Keep lags of power (past), weather, dust, calendar
leak_like = {"P_clean_W", "P_clean_norm"}  # concurrent clean power is strong proxy of current conditions
features_24h = [c for c in features_nowcast if c not in leak_like]

print(f"Nowcast features: {len(features_nowcast)}")
print(f"24h features:     {len(features_24h)}")
print("24h feature list:\n", features_24h)

Nowcast features: 49
24h features:     47
24h feature list:
 ['G_POA_Wm2', 'temp_amb_C', 'T_cell_C', 'RH_pct', 'wind_speed_ms', 'cloud_cover_pct', 'precipitation_mm', 'dust_density_reconstructed_PCHIP_gm2', 'is_daytime', 'hour', 'dayofweek', 'month', 'dayofyear', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'P_dusty_W_lag_1h', 'P_dusty_W_lag_3h', 'P_dusty_W_lag_6h', 'P_dusty_W_lag_12h', 'P_dusty_W_lag_24h', 'P_dusty_W_lag_48h', 'P_dusty_W_lag_72h', 'P_dusty_W_roll_mean_3h', 'P_dusty_W_roll_max_3h', 'P_dusty_W_roll_mean_6h', 'P_dusty_W_roll_max_6h', 'P_dusty_W_roll_mean_12h', 'P_dusty_W_roll_max_12h', 'P_dusty_W_roll_mean_24h', 'P_dusty_W_roll_max_24h', 'dust_density_reconstructed_PCHIP_gm2_lag_24h', 'dust_density_reconstructed_PCHIP_gm2_roll_mean_24h', 'dust_density_reconstructed_PCHIP_gm2_roll_max_24h', 'dust_sq', 'G_POA_Wm2_lag_1h', 'G_POA_Wm2_lag_3h', 'G_POA_Wm2_lag_6h', 'G_POA_Wm2_lag_24h', 'G_POA_Wm2_roll_mean_6h', 'G_POA_Wm2_roll_max_6h', 'G_POA_Wm2_roll_mean_24h', 'G_POA_Wm2_ro

## 8. Save artifacts

In [10]:
out_parquet = OUT_DIR / "forecast_ready.parquet"
model_df.to_parquet(out_parquet)

meta = {
    "n_rows": len(model_df),
    "features_nowcast": features_nowcast,
    "features_24h": features_24h,
    "target_nowcast": "target_nowcast",
    "target_24h": "target_24h",
    "splits": model_df["split"].value_counts().to_dict(),
    "train_end": str(model_df.loc[model_df.split=="train"].index.max()),
    "val_end": str(model_df.loc[model_df.split=="val"].index.max()),
    "test_end": str(model_df.index.max()),
    "notes": [
        "Source: Laghouat_2022_final.csv only",
        "Dust is PCHIP-reconstructed between gravimetric samples",
        "Chronological 70/15/15 split",
        "Daytime flag: G_POA > 5 W/m2 when available",
    ],
}

import json
with open(OUT_DIR / "preprocessing_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print("Saved:", out_parquet)
print("Saved:", OUT_DIR / "preprocessing_meta.json")
print("Done.")

Saved: processed_data\forecast_ready.parquet
Saved: processed_data\preprocessing_meta.json
Done.


## Summary

| Item | Choice |
|------|--------|
| Source | Laghouat CSV only |
| Split | Chronological 70/15/15 |
| Targets | nowcast + 24h-ahead |
| Leakage control | lags/rolling past-only; concurrent P_dusty excluded from features |
| Daytime | `G_POA > 5` |
| Limits | short season, reconstructed dust, single site |